# AMD Cloud 实例连接检查

本 Notebook 在运行它的电脑上使用系统 OpenSSH，检查地址解析、SSH 端口和密钥认证。它不会读取或显示私钥内容。

先复制 `instance.example` 为本地 `instance`，并按需复制 `local_settings.example.json` 为 `local_settings.json`。用你自己的实例 SSH 命令和密钥路径填写本地配置；这些本地文件已加入 `.gitignore`，不要提交。

先在 AMD Cloud 网页启动或获授权使用实例。首次连接前，在本地 PowerShell 交互式 SSH 连接并核对服务器指纹；确认后再运行本 Notebook。HTTP 检查只适用于实例另有 HTTP/HTTPS 服务，SSH 端口本身不是 HTTP 服务。代理检查为可选项，需要本机 HTTP/Mixed 代理已启动。

In [ ]:
from pathlib import Path
import json
import shlex
import shutil
import socket
import subprocess
import time

WORKSPACE = Path.cwd()
INSTANCE_FILE = WORKSPACE / "instance"
SETTINGS_FILE = WORKSPACE / "local_settings.json"
if not INSTANCE_FILE.is_file():
    raise FileNotFoundError("找不到本地 instance 文件；请复制 instance.example 为 instance 并填入实例 SSH 命令。")

parts = shlex.split(INSTANCE_FILE.read_text(encoding="utf-8").strip())
if len(parts) < 2 or parts[0] != "ssh":
    raise ValueError("instance 文件格式应类似：ssh user@host -p port")

destination = parts[1]
if "@" not in destination:
    raise ValueError("未能从 instance 文件解析 user@host")
SSH_USER, SSH_HOST = destination.rsplit("@", 1)
SSH_PORT = int(parts[parts.index("-p") + 1]) if "-p" in parts else 22
SSH_EXE = shutil.which("ssh")

settings = json.loads(SETTINGS_FILE.read_text(encoding="utf-8")) if SETTINGS_FILE.is_file() else {}
key_path = settings.get("identity_file", "~/.ssh/id_ed25519")
SSH_KEY = Path(key_path).expanduser() if key_path else None
SSH_KEY_OPTIONS = ["-i", str(SSH_KEY), "-o", "IdentitiesOnly=yes"] if SSH_KEY else []
HTTP_URL = str(settings.get("http_url", "")).strip()
LOCAL_PROXY_PORT = int(settings.get("local_proxy_port", 7890))
REMOTE_PROXY_PORT = int(settings.get("remote_proxy_port", 17890))
RUN_PROXY_CHECK = bool(settings.get("run_proxy_check", False))

CHECKS = {"DNS": None, "TCP": None, "SSH": None, "HTTP": None, "代理": None}
print(f"目标实例：{SSH_USER}@{SSH_HOST}:{SSH_PORT}")
print(f"本机 OpenSSH：{SSH_EXE or '未找到'}")
print(f"显式私钥文件有效：{SSH_KEY is None or SSH_KEY.is_file()}（不会读取或显示私钥内容）")

In [ ]:
try:
    address_info = socket.getaddrinfo(SSH_HOST, SSH_PORT, type=socket.SOCK_STREAM)
    resolved_addresses = sorted({item[4][0] for item in address_info})
    CHECKS["DNS"] = True
    print("DNS 解析成功：", ", ".join(resolved_addresses))
except socket.gaierror as exc:
    CHECKS["DNS"] = False
    print(f"DNS 解析失败：{exc}")

In [ ]:
try:
    with socket.create_connection((SSH_HOST, SSH_PORT), timeout=8):
        CHECKS["TCP"] = True
        print(f"TCP 连接成功：{SSH_HOST}:{SSH_PORT}")
except OSError as exc:
    CHECKS["TCP"] = False
    print(f"TCP 连接失败：{type(exc).__name__}: {exc}")
    print("确认实例正在运行、主机/端口无误，并检查本地网络是否允许出站连接。")

In [ ]:
if not SSH_EXE:
    CHECKS["SSH"] = False
    print("未找到 OpenSSH；请确认 Windows OpenSSH Client 已安装并在 PATH 中。")
elif SSH_KEY and not SSH_KEY.is_file():
    CHECKS["SSH"] = False
    print(f"找不到配置的私钥文件：{SSH_KEY}。请检查 local_settings.json 中的 identity_file。")
else:
    ssh_command = [
        SSH_EXE,
        *SSH_KEY_OPTIONS,
        "-o", "BatchMode=yes",
        "-o", "ConnectTimeout=8",
        "-p", str(SSH_PORT),
        f"{SSH_USER}@{SSH_HOST}",
        "printf 'SSH_AUTH_OK\\n'; whoami; hostname; uname -srm",
    ]
    try:
        ssh_result = subprocess.run(
            ssh_command, capture_output=True, text=True, timeout=20, errors="replace"
        )
        CHECKS["SSH"] = ssh_result.returncode == 0 and "SSH_AUTH_OK" in ssh_result.stdout
        if CHECKS["SSH"]:
            print("SSH 密钥认证成功；远端信息：")
            print(ssh_result.stdout.strip())
        else:
            print(f"SSH 登录失败，退出码：{ssh_result.returncode}")
            print((ssh_result.stderr or ssh_result.stdout).strip())
            print("确认首次连接的服务器指纹已接受，并检查公钥授权和本地身份密钥配置。")
    except subprocess.TimeoutExpired:
        CHECKS["SSH"] = False
        print("SSH 检查超时；确认实例运行中、地址可达，且 SSH 端口正确。")

In [ ]:
import urllib.error
import urllib.request

if not HTTP_URL:
    CHECKS["HTTP"] = None
    print("HTTP 检查已跳过：SSH 地址不是 HTTP 服务；如有 Web 服务，请在 local_settings.json 填写完整 http_url。")
else:
    request = urllib.request.Request(HTTP_URL, method="GET")
    started_at = time.perf_counter()
    try:
        with urllib.request.urlopen(request, timeout=10) as response:
            elapsed = time.perf_counter() - started_at
            preview = response.read(256).decode("utf-8", errors="replace")
            CHECKS["HTTP"] = True
            print(f"HTTP 请求完成：状态码 {response.status}，耗时 {elapsed:.2f} 秒")
            print("响应摘要：", repr(preview[:200]))
    except urllib.error.HTTPError as exc:
        elapsed = time.perf_counter() - started_at
        preview = exc.read(256).decode("utf-8", errors="replace")
        CHECKS["HTTP"] = True
        print(f"HTTP 服务有响应：状态码 {exc.code}，耗时 {elapsed:.2f} 秒")
        print("响应摘要：", repr(preview[:200]))
    except (urllib.error.URLError, TimeoutError, OSError) as exc:
        CHECKS["HTTP"] = False
        print(f"HTTP 请求失败：{type(exc).__name__}: {exc}")

In [ ]:
if not RUN_PROXY_CHECK:
    CHECKS["代理"] = None
    print("代理检查已跳过。确认本机 HTTP/Mixed 代理已启动后，可在 local_settings.json 中将 run_proxy_check 改为 true。")
elif not SSH_EXE:
    CHECKS["代理"] = False
    print("未找到 OpenSSH；无法建立反向代理隧道。")
elif SSH_KEY and not SSH_KEY.is_file():
    CHECKS["代理"] = False
    print(f"找不到配置的私钥文件：{SSH_KEY}。请检查 local_settings.json。")
else:
    try:
        with socket.create_connection(("127.0.0.1", LOCAL_PROXY_PORT), timeout=3):
            pass
        remote_proxy_command = (
            f"export http_proxy=http://127.0.0.1:{REMOTE_PROXY_PORT} "
            f"https_proxy=http://127.0.0.1:{REMOTE_PROXY_PORT} "
            f"HTTP_PROXY=http://127.0.0.1:{REMOTE_PROXY_PORT} "
            f"HTTPS_PROXY=http://127.0.0.1:{REMOTE_PROXY_PORT}; "
            "curl -sSIL --max-time 20 -o /dev/null -w 'HTTP %{http_code}\\n' https://github.com"
        )
        proxy_command = [
            SSH_EXE,
            *SSH_KEY_OPTIONS,
            "-o", "BatchMode=yes",
            "-o", "ConnectTimeout=8",
            "-o", "ExitOnForwardFailure=yes",
            "-R", f"{REMOTE_PROXY_PORT}:127.0.0.1:{LOCAL_PROXY_PORT}",
            "-p", str(SSH_PORT),
            f"{SSH_USER}@{SSH_HOST}",
            remote_proxy_command,
        ]
        proxy_result = subprocess.run(
            proxy_command, capture_output=True, text=True, timeout=40, errors="replace"
        )
        CHECKS["代理"] = proxy_result.returncode == 0
        if CHECKS["代理"]:
            print("反向代理测试成功：", proxy_result.stdout.strip())
            print("这是一次性测试；SSH 进程退出后转发也会关闭。")
        else:
            print(f"反向代理测试失败，退出码：{proxy_result.returncode}")
            print((proxy_result.stderr or proxy_result.stdout).strip())
    except OSError as exc:
        CHECKS["代理"] = False
        print(f"本机代理端口 {LOCAL_PROXY_PORT} 无法连接：{exc}")
    except subprocess.TimeoutExpired:
        CHECKS["代理"] = False
        print("代理转发测试超时；确认代理可用，并检查云端是否允许 SSH 反向端口转发。")

In [ ]:
status_labels = {True: "通过", False: "失败", None: "跳过 / 未运行"}
print("检查项\t结果")
print("--------\t------------")
for name, value in CHECKS.items():
    print(f"{name}\t{status_labels[value]}")

## 常见结果

- **DNS / TCP 超时或拒绝**：确认实例已运行，并核对本地 `instance` 中的主机和端口。
- **`Host key verification failed`**：先在 PowerShell 交互式连接，核对服务器指纹后接受；不要关闭 host-key 校验来绕过错误。
- **`Permission denied (publickey)`**：检查 `local_settings.json` 中的 `identity_file` 是否指向与已授权公钥配对的本机私钥；若加入现有实例，确认队长已按官方 FAQ 添加你的公钥。
- **仅代理检查失败**：确认本机代理为 HTTP/Mixed 且监听配置的端口。检查单元会建立一次性 `-R` 转发，测试结束隧道关闭；实际工作时须保持 SSH 隧道运行。